# 02 — Model Training & SHAP Analysis

## UPI Propensity Score API

**Objective:** Train an XGBoost classifier on the engineered features, evaluate with Precision@K (the right metric for campaign targeting), generate SHAP explanations, and save model artifacts.

**Inputs:** `data/processed/features.parquet`, `models/feature_names.json`  
**Outputs:** `models/xgb_model.pkl`, `models/model_metadata.json`, `models/feature_defaults.json`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import pickle
import time
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from pathlib import Path
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

DATA_PROCESSED = Path('../data/processed')
MODELS_DIR = Path('../models')

## 1. Load Data

In [ ]:
df = pd.read_parquet(DATA_PROCESSED / 'features.parquet')

with open(MODELS_DIR / 'feature_names.json') as f:
    FEATURE_COLS = json.load(f)

X = df[FEATURE_COLS]
y = df['will_transact']

print(f"Dataset: {X.shape}")
print(f"Features: {len(FEATURE_COLS)}")
print(f"Target rate: {y.mean():.2%}")

## 2. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Train target rate: {y_train.mean():.2%} | Test target rate: {y_test.mean():.2%}")

## 3. Handle Class Imbalance

`scale_pos_weight` adjusts the loss function directly — cleaner than SMOTE for tree-based models and avoids synthetic sample artifacts.

In [ ]:
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

print(f"Negative (dormant): {neg_count:,}")
print(f"Positive (will transact): {pos_count:,}")
print(f"scale_pos_weight: {scale_pos_weight:.1f}")

## 4. Train XGBoost

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    early_stopping_rounds=20,
    random_state=42,
    n_jobs=-1
)

train_start = time.time()
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)
train_time = time.time() - train_start

print(f"\nTraining time: {train_time:.1f}s")
print(f"Best iteration: {model.best_iteration}")

## 5. Evaluate — Standard Metrics

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

auc = roc_auc_score(y_test, y_prob)
print(f"AUC-ROC: {auc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['dormant', 'will_transact']))

## 6. Evaluate — Precision@K

**This is the key metric for propensity targeting.** AUC tells you global separation quality. Precision@K answers: *"If I target the top K% of users, what fraction will actually convert?"* — which is what determines your campaign ROI.

In [ ]:
def precision_at_k(y_true, y_scores, k):
    """Out of top K ranked users, what fraction are truly positive?"""
    top_k_idx = np.argsort(y_scores)[::-1][:k]
    return y_true.iloc[top_k_idx].mean()


test_size = len(y_test)
baseline = y_test.mean()
results = []

print(f"{'Metric':<20} {'Value':>8} {'Baseline':>10} {'Lift':>8}")
print('-' * 50)

for k_pct in [0.01, 0.05, 0.10, 0.20]:
    k = int(test_size * k_pct)
    p_at_k = precision_at_k(y_test, y_prob, k)
    lift = p_at_k / baseline
    label = f"Precision@{int(k_pct*100)}%"
    print(f"{label:<20} {p_at_k:>8.3f} {baseline:>10.3f} {lift:>7.1f}x")
    results.append({'metric': label, 'value': round(p_at_k, 4), 'lift': round(lift, 1)})

# Vectorized precision curve (much faster than per-k argsort loop)
sorted_idx = np.argsort(y_prob)[::-1]
y_sorted = y_test.iloc[sorted_idx].values
cumsum = np.cumsum(y_sorted)

k_steps = np.arange(500, test_size, 500)  # sample every 500
precisions = cumsum[k_steps] / (k_steps + 1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_steps / test_size * 100, precisions, 'b-', linewidth=2)
ax.axhline(y=baseline, color='r', linestyle='--', label=f'Baseline ({baseline:.3f})')
ax.set_xlabel('Top K% of users targeted')
ax.set_ylabel('Precision (conversion rate in targeted group)')
ax.set_title('Precision@K Curve — Campaign Targeting Effectiveness')
ax.legend()
plt.tight_layout()
plt.savefig('precision_at_k.png', dpi=100)
plt.show()

## 7. SHAP Explanations

In [ ]:
# TreeExplainer is fast for XGBoost
explainer = shap.TreeExplainer(model)

# SHAP on a sample for the global summary plot
shap_sample = X_test.iloc[:1000]

shap_start = time.time()
shap_values = explainer.shap_values(shap_sample)
shap_time = time.time() - shap_start

print(f"SHAP computation time for 1000 samples: {shap_time:.2f}s")
print(f"Estimated time for 5000 samples: {shap_time * 5:.1f}s")
print(f"\nNote: For the batch API endpoint, SHAP is optional (include_shap parameter)")
print(f"to avoid timeouts on large batches.")

# Global summary plot
shap.summary_plot(shap_values, shap_sample, feature_names=FEATURE_COLS, show=False)
plt.savefig('shap_summary.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Per-prediction explanation example (what the API returns)
def get_top_shap_features(shap_values_row, feature_names, top_n=3):
    """Returns top N features driving a single prediction."""
    impacts = list(zip(feature_names, shap_values_row))
    impacts.sort(key=lambda x: abs(x[1]), reverse=True)
    return [
        {
            "feature": name,
            "impact": round(float(val), 4),
            "direction": "increases_propensity" if val > 0 else "decreases_propensity"
        }
        for name, val in impacts[:top_n]
    ]

# Demo on first test sample
sample_shap = explainer.shap_values(X_test.iloc[[0]])
sample_score = model.predict_proba(X_test.iloc[[0]])[0][1]
top_features = get_top_shap_features(sample_shap[0], FEATURE_COLS)

print(f"Sample prediction score: {sample_score:.4f}")
print(f"Top 3 drivers:")
for feat in top_features:
    print(f"  {feat['feature']}: {feat['impact']:+.4f} ({feat['direction']})")

## 8. Save Model Artifacts

In [ ]:
# Save trained model
with open(MODELS_DIR / 'xgb_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print(f"Model saved to {MODELS_DIR / 'xgb_model.pkl'}")

# Save feature defaults (medians from training data for API inference)
feature_defaults = {}
for col in FEATURE_COLS:
    if X_train[col].dtype in ['float64', 'float32', 'int64', 'int32']:
        feature_defaults[col] = float(X_train[col].median())
    else:
        feature_defaults[col] = 0.0

with open(MODELS_DIR / 'feature_defaults.json', 'w') as f:
    json.dump(feature_defaults, f, indent=2)
print(f"Feature defaults saved to {MODELS_DIR / 'feature_defaults.json'}")

# Save model metadata for lightweight versioning
metadata = {
    "trained_at": datetime.now().isoformat(),
    "n_features": len(FEATURE_COLS),
    "n_train_samples": len(X_train),
    "n_test_samples": len(X_test),
    "auc_roc": round(auc, 4),
    "best_iteration": model.best_iteration,
    "scale_pos_weight": round(scale_pos_weight, 2),
    "xgboost_version": xgb.__version__,
    "precision_at_k": results,
    "training_time_seconds": round(train_time, 1),
    "shap_time_1000_samples": round(shap_time, 2)
}

with open(MODELS_DIR / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata saved to {MODELS_DIR / 'model_metadata.json'}")

print(f"\n--- Summary ---")
print(f"AUC-ROC: {auc:.4f}")
for r in results:
    print(f"{r['metric']}: {r['value']} ({r['lift']}x lift)")